# Reproduction Notebook

This notebook is a step-by-step path to reproduce **Figures 1 and 2** of the SILAGE paper. They are two synthetic optimization-trajectory grids:

- **Figure 1** — the $m \ge n$ regime, $(n,m)=(50,250)$; in the code this is the setting `m_gt_n` (SILAGE Algorithm 1).
- **Figure 2** — the $n > m$ regime, $(n,m)=(250,50)$; in the code this is the setting `n_gt_m` (SILAGE Algorithm 2).

Here $n$ is the number of groups/blocks/silos and $m$ the number of samples per group ($N=nm$ total), matching the nested objective $f(x)=\frac{1}{n}\sum_{i=1}^{n}\frac{1}{m}\sum_{j=1}^{m} f_{i,j}(x)$ in the paper. Each figure is a $2\times2$ grid over the four $(\delta_1,\delta_2)$ heterogeneity regimes. The full code-to-paper symbol mapping used below is in the **Notation & naming conventions** section of the [README](../README.md).

To draw these figures for the paper, we first ran grid searches over the batch-size grids specified in the paper and then plotted the tuned choices. This repository does not re-run that grid search in the main reproduction path. Instead, it follows the recorded workflow: generate the synthetic data and constants, launch the methods at the recorded tuned parameters, and draw the trajectory grids.

## 1. Create the Environment

Use one of the two environments below, depending on what you want to reproduce.

### Linux/CUDA full-run environment

This is the original environment used for the experiment runs. It pins PyTorch with CUDA 11.8 and is intended for Linux machines with NVIDIA GPUs:

```bash
conda env create -f environment.yml
conda activate silage
```

### macOS/CPU plotting environment

Use this environment on macOS when you only need to run this notebook and redraw Figures 1 and 2 from the bundled logs. It intentionally omits `pytorch-cuda`, because native macOS does not provide CUDA support for PyTorch:

```bash
conda env create -f environment_plotting.yml
conda activate silage-plot
python -m ipykernel install --user --name silage-plot --display-name "Python (silage-plot)"
```

The original experiment scripts were configured for a CUDA machine with visible devices `cuda:1` and `cuda:0`; preprocessing defaults to `cuda:0`. If CUDA is unavailable, preprocessing automatically falls back to CPU, and the cells below also expose an explicit `--device cpu` path. CPU-only execution is possible in the code path but can be much slower than the CUDA workstation used for the paper. For a faithful full rerun of the experiment jobs, use the Linux/CUDA environment above.


In [ ]:
from pathlib import Path
import os
import shlex
import shutil
import subprocess
import sys

START_DIR = Path.cwd().resolve()
repo_candidates = [START_DIR, *START_DIR.parents]
REPO_ROOT = next((path for path in repo_candidates if (path / 'environment.yml').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate environment.yml in this directory or any parent directory.')

if (START_DIR / 'run_synthetic_experiments_local.py').exists():
    LOGREG_DIR = START_DIR
else:
    candidate_dirs = [
        REPO_ROOT / 'experiments_open' / 'logistic_regression',
        REPO_ROOT / 'experiments' / 'logistic_regression',
    ]
    LOGREG_DIR = next((path for path in candidate_dirs if (path / 'run_synthetic_experiments_local.py').exists()), None)
    if LOGREG_DIR is None:
        raise FileNotFoundError('Could not locate the logistic-regression experiment directory.')

os.chdir(LOGREG_DIR)
if str(LOGREG_DIR) not in sys.path:
    sys.path.insert(0, str(LOGREG_DIR))

try:
    import torch
    CUDA_DEVICE_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
    DEVICE = 'cuda:0' if CUDA_DEVICE_COUNT > 0 else 'cpu'
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('CUDA device count:', CUDA_DEVICE_COUNT)
    if CUDA_DEVICE_COUNT > 0:
        print('Selected device override:', DEVICE, torch.cuda.get_device_name(0))
    else:
        print('Selected device override:', DEVICE)
except Exception as exc:
    CUDA_DEVICE_COUNT = 0
    DEVICE = 'cpu'
    print('Could not import torch while selecting a device; defaulting to cpu.')
    print(type(exc).__name__, exc)

TMUX_AVAILABLE = shutil.which('tmux') is not None

print('Repository root:', REPO_ROOT)
print('Working directory:', LOGREG_DIR)
print('tmux available:', TMUX_AVAILABLE)


## 2. Generate Synthetic Data and Preprocessing Constants

Run the full synthetic preprocessing pass:

```bash
python run_preprocessing_synthetic_local.py
```

This creates the full `2 x 4` synthetic grid from scratch: two **size settings** and four **heterogeneity regimes** (the same ones from Section 2 of the paper). In the code we name them as follows:

- size settings — `m_gt_n` is the $m \ge n$ regime $(n,m)=(50,250)$; `n_gt_m` is the $n > m$ regime $(n,m)=(250,50)$.
- heterogeneity regimes — `d1` $\equiv \delta_1$ (across-group similarity, Assumption 3) and `d2` $\equiv \delta_2$ (within-group similarity, Assumption 4): in the code we call them `d1_small_d2_small`, `d1_small_d2_large`, `d1_large_d2_small`, `d1_large_d2_large`, i.e. $(\delta_1\text{ small/large}, \delta_2\text{ small/large})$.

It also computes the comp-param constants that calibrate each method's theoretically admissible stepsize (paper Remark 1). In the code we call them, with the paper symbol in parentheses:

- `lambda_reg` ($\lambda$, regularization weight, $=200$), `delta1_emp` ($\delta_1$), `delta2_emp` ($\delta_2$), `delta_flat_emp` ($\delta_{\mathrm{flat}}$, flattened similarity;
- `L_ij_max_emp` ($L_{\max}=\max_{i,j}L_{i,j}$, eq. (6)), `L_i_max_emp` ($\max_i L_i$, eq. (5)), `L_global_emp` ($L$, smoothness of the averaged objective $f$, Assumption 2).

The `_emp` keys are empirical (probe-set) estimates; the saved `_wc` keys are their worst-case diagnostic counterparts. The next cell prints these per `(setting, regime)`; the $\delta_1,\delta_2,L,L_{\max}$ values reproduce **Table 4** of the paper.

`lambda_reg` is set by the synthetic regime table in `src/synthetic_logreg.py`; for these regimes it is `200.0`. To force CPU, run `python run_preprocessing_synthetic_local.py --device cpu`.

In [ ]:
RUN_PREPROCESSING = False  # Set to True to execute this step.

cmd = [sys.executable, 'run_preprocessing_synthetic_local.py']
if DEVICE != 'cuda:0':
    cmd += ['--device', DEVICE]

print(' '.join(shlex.quote(part) for part in cmd))

if RUN_PREPROCESSING:
    subprocess.run(cmd, cwd=LOGREG_DIR, check=True)


In [ ]:
RUN_CHECK_COMP_PARAMS = False  # Requires preprocessing output (workflow B). After running run_preprocessing_synthetic_local.py, set True to print the Table 4 constants.

import numpy as np

from src.synthetic_logreg import (
    SYNTHETIC_DIRICHLET_LOGREG_DATASET,
    synthetic_logreg_extension_suffix,
    synthetic_logreg_regime_params,
    synthetic_logreg_setting_to_nm,
)
from src.utils import load_comp_params_bundle

computed_param_keys = [
    'lambda_reg',
    'dim',
    'n_groups',
    'm_per_group',
    'delta1_emp',
    'delta2_emp',
    'delta_flat_emp',
    'L_ij_max_emp',
    'L_i_max_emp',
    'L_global_emp',
    'delta1_wc',
    'delta2_wc',
    'L_ij_max_wc',
    'L_i_max_wc',
    'L_global_wc',
]

synthetic_settings = ['m_gt_n', 'n_gt_m']
synthetic_regimes = [
    'd1_small_d2_small',
    'd1_small_d2_large',
    'd1_large_d2_small',
    'd1_large_d2_large',
]


def comp_params_dir_for_case(setting, regime, d=1000):
    n_groups, m_per_group = synthetic_logreg_setting_to_nm(setting)
    regime_params = synthetic_logreg_regime_params(setting, regime)
    suffix = synthetic_logreg_extension_suffix(
        setting=setting,
        regime=regime,
        n_groups=n_groups,
        m_per_group=m_per_group,
        d=d,
        K=int(regime_params['K']),
        T=int(regime_params['T']),
    )
    return (
        LOGREG_DIR
        / f'data_{SYNTHETIC_DIRICHLET_LOGREG_DATASET}'
        / f'comp_params_log-reg_{SYNTHETIC_DIRICHLET_LOGREG_DATASET}{suffix}'
    )


def format_param_value(value):
    if isinstance(value, np.ndarray):
        if value.shape == ():
            value = value.item()
        else:
            return np.array2string(value, precision=6, threshold=8)
    if isinstance(value, float):
        return f'{value:.8g}'
    return str(value)


if not RUN_CHECK_COMP_PARAMS:
    print('Skipping comp-param check (RUN_CHECK_COMP_PARAMS=False).')
    print('These constants come from preprocessing (workflow B); the Section 4 figures do not need them.')
    print('After running run_preprocessing_synthetic_local.py, set RUN_CHECK_COMP_PARAMS=True to print the Table 4 constants.')
else:
    for setting in synthetic_settings:
        for regime in synthetic_regimes:
            comp_params_dir = comp_params_dir_for_case(setting, regime)
            try:
                bundle = load_comp_params_bundle(str(comp_params_dir), computed_param_keys, is_print=0)
            except FileNotFoundError:
                raise FileNotFoundError(
                    f'Comp-params for [{setting} | {regime}] not found at {comp_params_dir}. '
                    'Run preprocessing first (workflow B): python run_preprocessing_synthetic_local.py'
                )
            missing = [key for key in computed_param_keys if key not in bundle]
            if missing:
                raise FileNotFoundError(f'Missing {missing} in {comp_params_dir}')
            print(f'[{setting} | {regime}]')
            for key in computed_param_keys:
                print(f'  {key}: {format_param_value(bundle[key])}')


## 3. Launch the Tuned Experiment Runs

Launch the methods used in Figures 1 and 2 at the recorded tuned parameters:

```bash
python run_synthetic_experiments_local.py --algorithm silage_m_gt_n --launch_mode tmux
python run_synthetic_experiments_local.py --algorithm silage_n_gt_m --launch_mode tmux
python run_synthetic_experiments_local.py --algorithm zerosarah --launch_mode tmux
python run_synthetic_experiments_local.py --algorithm d_zerosarah --launch_mode tmux
python run_synthetic_experiments_local.py --algorithm silver --launch_mode tmux
```

These commands use the tuned values recorded in `run_synthetic_experiments_local.py` and selected by the plotting cell below. The batch-size grid search used to obtain these values is not part of this reproduction path.

For CPU or a single visible CUDA device, pass `--device cpu` or `--device cuda:0`; the launcher will run one `tmux` job at a time for that explicit device. For a slow fully sequential debug run, use `--launch_mode direct --job_index <idx>`.


In [ ]:
RUN_EXPERIMENTS = False  # Set to True to launch the tuned experiments.

algorithms = [
    'silage_m_gt_n',
    'silage_n_gt_m',
    'zerosarah',
    'd_zerosarah',
    'silver',
]

device_args = [] if CUDA_DEVICE_COUNT >= 2 else ['--device', DEVICE]
experiment_commands = [
    [
        sys.executable,
        'run_synthetic_experiments_local.py',
        '--algorithm',
        algorithm,
        '--launch_mode',
        'tmux',
        *device_args,
    ]
    for algorithm in algorithms
]

for cmd in experiment_commands:
    print(' '.join(shlex.quote(part) for part in cmd))

if RUN_EXPERIMENTS:
    for cmd in experiment_commands:
        subprocess.run(cmd, cwd=LOGREG_DIR, check=True)


In [ ]:
if not TMUX_AVAILABLE:
    print('tmux is not available in this shell.')
else:
    result = subprocess.run(['tmux', 'ls'], cwd=LOGREG_DIR, check=False, capture_output=True, text=True)
    if result.returncode == 0:
        print(result.stdout)
    else:
        print(result.stderr.strip() or 'No active tmux sessions.')


## 4. Draw Figures 1 and 2

Run the cell below to draw the two trajectory grids. The **bundled run logs** under `logs/` are sufficient — preprocessing is not required for plotting. Each grid is a $2\times2$ panel over the four $(\delta_1,\delta_2)$ regimes; the panel titles show the qualitative regime label $(\delta_1\text{ small/large}, \delta_2\text{ small/large})$ and the suptitle shows $n,m$. (The measured constants $\delta_1,\delta_2,L,L_{\max}$ for each panel are in Table 4 of the paper and are reproduced by the check cell in Section 2.)

The outputs are saved to:

- `plots/synthetic_trajectory_grids/trajectory_grid_m_gt_n.pdf` (Figure 1, $m \ge n$)
- `plots/synthetic_trajectory_grids/trajectory_grid_n_gt_m.pdf` (Figure 2, $n > m$)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from src.plotting import create_synthetic_trajectory_grids

synthetic_plot_style = {
    'marker_size': 15,
    'label_fontsize': 25,
    'tick_labelsize': 25,
    'axis_labelsize': 25,
    'legend_fontsize': 25,
    'subplot_title_fontsize': 25,
    'suptitle_fontsize': 25,
}

trajectory_epoch_tick_step_by_setting = {
    'm_gt_n': {'d1_small_d2_small': 5},
    'n_gt_m': {
        'd1_small_d2_small': 5,
        'd1_small_d2_large': 5,
        'd1_large_d2_small': 5,
        'd1_large_d2_large': 5,
    },
}

trajectory_epoch_xlim_by_setting = {
    'n_gt_m': {
        'd1_small_d2_small': 20,
        'd1_small_d2_large': 20,
        'd1_large_d2_small': 25,
        'd1_large_d2_large': 25,
    },
    'm_gt_n': {
        'd1_small_d2_small': 25,
        'd1_small_d2_large': 40,
        'd1_large_d2_small': 25,
        'd1_large_d2_large': 40,
    },
}

trajectory_methods_by_setting = {
    'm_gt_n': ['SILAGE_m>n', 'ZeroSARAH', 'SILVER', 'D-ZeroSARAH'],
    'n_gt_m': ['SILAGE_n>m', 'ZeroSARAH', 'SILVER', 'D-ZeroSARAH'],
}

trajectory_run_selector = {
    'm_gt_n': {
        'd1_small_d2_small': {
            'SILAGE_m>n': {},
            'ZeroSARAH': {'batch_size': 192},
            'SILVER': {'batch_size': 46},
            'D-ZeroSARAH': {'client_subset_size': 1, 'batch_size': 6},
        },
        'd1_small_d2_large': {
            'SILAGE_m>n': {},
            'ZeroSARAH': {'batch_size': 192},
            'SILVER': {'batch_size': 128},
            'D-ZeroSARAH': {'client_subset_size': 31, 'batch_size': 1},
        },
        'd1_large_d2_small': {
            'SILAGE_m>n': {},
            'ZeroSARAH': {'batch_size': 11},
            'SILVER': {'batch_size': 96},
            'D-ZeroSARAH': {'client_subset_size': 31, 'batch_size': 1},
        },
        'd1_large_d2_large': {
            'SILAGE_m>n': {},
            'ZeroSARAH': {'batch_size': 11},
            'SILVER': {'batch_size': 96},
            'D-ZeroSARAH': {'client_subset_size': 31, 'batch_size': 1},
        },
    },
    'n_gt_m': {
        'd1_small_d2_small': {
            'SILAGE_n>m': {'batch_size': 6},
            'ZeroSARAH': {'batch_size': 192},
            'SILVER': {'batch_size': 46},
            'D-ZeroSARAH': {'client_subset_size': 41, 'batch_size': 1},
        },
        'd1_small_d2_large': {
            'SILAGE_n>m': {'batch_size': 1},
            'ZeroSARAH': {'batch_size': 31},
            'SILVER': {'batch_size': 128},
            'D-ZeroSARAH': {'client_subset_size': 96, 'batch_size': 1},
        },
        'd1_large_d2_small': {
            'SILAGE_n>m': {'batch_size': 1},
            'ZeroSARAH': {'batch_size': 31},
            'SILVER': {'batch_size': 96},
            'D-ZeroSARAH': {'client_subset_size': 96, 'batch_size': 1},
        },
        'd1_large_d2_large': {
            'SILAGE_n>m': {'batch_size': 1},
            'ZeroSARAH': {'batch_size': 31},
            'SILVER': {'batch_size': 50},
            'D-ZeroSARAH': {'client_subset_size': 96, 'batch_size': 1},
        },
    },
}

trajectory_outputs = create_synthetic_trajectory_grids(
    {
        'settings': ['m_gt_n', 'n_gt_m'],
        'plot_style': synthetic_plot_style,
        'trajectory_epoch_tick_step_by_setting': trajectory_epoch_tick_step_by_setting,
        'trajectory_epoch_xlim_by_setting': trajectory_epoch_xlim_by_setting,
        'trajectory_ymin': 1e-12,
        'trajectory_legend_bbox_y': 0.02,
        'trajectory_tight_layout_rect': [0, 0.10, 1, 0.95],
        'methods_by_setting': trajectory_methods_by_setting,
        'trajectory_run_selector': trajectory_run_selector,
        'factor': 1.0,
        'output_dir': 'plots/synthetic_trajectory_grids',
        'save_plot': 1,
        'show_plot': 1,
    }
)
trajectory_outputs
